In [2]:
import math
import numpy as np

In [3]:
vin_min = 4*2.5 #V
vin_typ = 4 *3.6 #V
vin_max = 4*4.2 #V

target_vout = 22 #V

target_iout_max = 8 #A
rout = target_vout/target_iout_max
print(rout)

target_fsw = 350 * 10**3 #Hz

target_inductor_ripple = 0.3*target_iout_max

calculated_inductance = vin_typ*(target_vout - vin_typ)/(target_inductor_ripple*target_fsw*target_vout)
print(f"calculated inductance is {round(calculated_inductance*10**6, 2)} uH") #uH
actual_inductance = 6.8 * 10**-6 #H




2.75
calculated inductance is 5.92 uH


In [4]:
#vref calculator

def vref_calculator(vout, vref, rbot):
    rtop = rbot*(vout-vref)/vref
    return rtop

def vout_calculator(vref, rbot, rtop):
    vout = vref/(rbot/(rtop+rbot))
    return vout

In [10]:
# Boost FB

vref_boost = 1.2
rbot_boost = 2200

rtop = vref_calculator(target_vout, vref_boost, rbot_boost)
print(f"Ideal top resistor is {round(rtop, 2)} Ohms")
rtop_actual_boost = 38000

vout_actual_boost = vout_calculator(vref_boost, rbot_boost, rtop_actual_boost)
print(f"Actual output voltage is {round(vout_actual_boost, 2)} V")

Ideal top resistor is 38133.33 Ohms
Actual output voltage is 21.93 V


In [6]:
# Equivalent Capacitance Function Definition
# https://www.power-mag.com/pdf/feature_pdf/1387888355_Murata_Feature_Layout_1.pdf

#Returned datastructure
class EquivalentCapacitance:
    def __init__(self, capacitances, fsw, ripple_current, equivalent_capacitance, equivalent_esr, ripple_voltage):
        self.capacitances = capacitances # Will be of the form [qty, capacitance (F), esr (Ohm), ripple current (A)] where the 4th column will only be added if a total ripple current was provided
        self.fsw = fsw
        self.ripple_current = ripple_current
        self.equivalent_capacitance = equivalent_capacitance
        self.equivalent_esr = equivalent_esr
        self.ripple_voltage = ripple_voltage

    def print(self):
        print(f"Capacitance Matrix: {self.capacitances}")
        print(f"Switching Frequency: {self.fsw} Hz")
        print(f"Ripple Current: {self.ripple_current} A")
        print(f"Equivalent Capacitance: {self.equivalent_capacitance} F")
        print(f"Equivalent ESR: {self.equivalent_esr} Ohms")
        print(f"Ripple Voltage: {self.ripple_voltage} V")

def equiv_capacitance(capacitances, fsw, current = 0):
    # capacitances is a 3 column, n row matrix with rows of the form [qty, capacitance (uF), esr (mOhm)]
    # fsw is the fundamental switching frequency (kHz)
    # current is optional. If provided, the return will include the ripple voltage and current per capacitor
    
    cs = np.array(capacitances)
    cs = cs.astype(float)
    xs = 1/(2*3.1416*fsw*cs[:, 1])
    
    cp = cs.copy()
    cp[:, 1] = cs[:, 1]/(1 + (cs[:, 2]/xs)**2)
    cp[:, 2] = (cs[:, 2]**2 + xs**2)/cs[:, 2]
    
    cpe = np.sum(cp[:, 0]*cp[:, 1])
    rpe = 1/np.sum(cp[:, 0]/cp[:, 2])

    xpe = 1/(2*3.1416*fsw*cpe)
    cse = cpe*(1 + (xpe/rpe)**2)
    rse = rpe/(1 + (rpe/xpe)**2)
    xse = 1/(2*3.1416*fsw*cse)

    dv = np.nan
    if(current != 0):
        dv = current*math.sqrt(rse**2 + xse**2)
        di = dv/np.sqrt(cs[:, 2]**2 + xs**2)
        cs = np.hstack((cs, di.reshape(-1, 1)))
    return EquivalentCapacitance(cs, fsw, current if current > 0 else np.nan, cse, rse, dv)

In [7]:
# Effective Capacitance Calculations

actual_cout = np.array([[1, 0.07, 100], #100nF
                        [1, 0.15, 12], #1uF
                        [3, 2.2, 2], #22uF
                        [3, 220, 80]]) #220u
actual_cout[:, 1] = actual_cout[:, 1]*10**-6 # Convert to F
actual_cout[:, 2] = actual_cout[:, 2]*10**-3 # Convert to Ohms
print(actual_cout)

capacitances = equiv_capacitance(actual_cout, target_fsw)
effective_capacitance = capacitances.equivalent_capacitance
effective_esr = capacitances.equivalent_esr
print(f"Effective Capacitance is {round(effective_capacitance*10**6, 2)} uF")
print(f"Effective Resistance is {round(effective_esr*10**3, 2)} mOhms")

[[1.0e+00 7.0e-08 1.0e-01]
 [1.0e+00 1.5e-07 1.2e-02]
 [3.0e+00 2.2e-06 2.0e-03]
 [3.0e+00 2.2e-04 8.0e-02]]
Effective Capacitance is 47.57 uF
Effective Resistance is 22.53 mOhms


In [8]:
# Compensation Calculations

shunt_resistance = 0.003 #ohms
g_comp = 0.006 #V/V
g_m = 2*10**-3 #mho

duty_cycle_max = 1-(vin_min/vout_actual_boost)
duty_cycle_typ = 1-(vin_typ/vout_actual_boost)

f_z_rhp = (rout*(1-duty_cycle_max)**2)/(2*3.1416*actual_inductance)
fxo = min(target_fsw/10, f_z_rhp/5)
print(f"Crossover Frequency is {round(fxo*10**-3, 2)} kHz")

r_comp = (2*3.1416*effective_capacitance*shunt_resistance*vout_actual_boost**2*fxo)/(g_comp*g_m*vin_min)
print(f"The Compensation Resistance is {round(r_comp*10**-3, 2)} kOhms")
r_comp_actual = r_comp

c_comp = math.sqrt((effective_capacitance*rout)/(4*3.1416*r_comp_actual**2*fxo))
print(f"The Compensation Capacitance is {round(c_comp*10**9, 2)} nF")
c_comp_actual = c_comp

c_hf = (c_comp_actual*actual_inductance)/((c_comp_actual*(1-duty_cycle_typ)**2*rout*r_comp_actual)-actual_inductance)
print(f"The High Frequency Capacitance is {round(c_hf*10**12, 2)} pF")

Crossover Frequency is 2.53 kHz
The Compensation Resistance is 9.62 kOhms
The Compensation Capacitance is 6.67 nF
The High Frequency Capacitance is 695.94 pF
